# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [3]:

print(fraud['fraud'].value_counts())

print("\nPercentage distribution:")
print(fraud['fraud'].value_counts(normalize=True) * 100)


0.0    912597
1.0     87403
Name: fraud, dtype: int64

Percentage distribution:
0.0    91.2597
1.0     8.7403
Name: fraud, dtype: float64


In [4]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X = fraud.drop(columns=['fraud'])
y = fraud['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully!")


Model trained successfully!


In [5]:

from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000


Confusion Matrix:
[[181296   1223]
 [  6895  10586]]


In [6]:

from sklearn.utils import resample

X_majority = X_train[y_train == 0]
X_minority = X_train[y_train == 1]

X_minority_up = resample(
    X_minority,
    replace=True,
    n_samples=len(X_majority),
    random_state=42
)

X_train_over = np.vstack((X_majority, X_minority_up))

y_train_over = np.concatenate([
    np.zeros(len(X_majority)),
    np.ones(len(X_minority_up))
])

model_over = LogisticRegression(max_iter=1000)
model_over.fit(X_train_over, y_train_over)

y_pred_over = model_over.predict(X_test)

print(classification_report(y_test, y_pred_over))
print(confusion_matrix(y_test, y_pred_over))


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

[[170391  12128]
 [   910  16571]]


In [7]:

from sklearn.utils import resample

X_majority_down = resample(
    X_majority,
    replace=False,
    n_samples=len(X_minority),
    random_state=42
)

X_train_under = np.vstack((X_majority_down, X_minority))

y_train_under = np.concatenate([
    np.zeros(len(X_majority_down)),
    np.ones(len(X_minority))
])

model_under = LogisticRegression(max_iter=1000)
model_under.fit(X_train_under, y_train_under)

y_pred_under = model_under.predict(X_test)

print(classification_report(y_test, y_pred_under))
print(confusion_matrix(y_test, y_pred_under))


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

[[170394  12125]
 [   918  16563]]


In [8]:
%pip install imbalanced-learn==0.9.1

Note: you may need to restart the kernel to use updated packages.


In [10]:

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)

print(classification_report(y_test, y_pred_smote))
print(confusion_matrix(y_test, y_pred_smote))


/Users/fatmacakir/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

[[170386  12133]
 [   907  16574]]


In [ ]:
### Conclusion

The original Logistic Regression model achieved 96% accuracy but detected only 61% of fraudulent transactions.

After applying oversampling, undersampling, and SMOTE, fraud recall increased to approximately 95%, although overall accuracy decreased to 93%.

SMOTE produced the fewest missed fraudulent transactions (907), making it the best option among the tested methods when detecting fraud is the priority.

However, all three balancing methods increased false positives. Therefore, recall and precision are more informative than accuracy alone for this imbalanced dataset.